In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from skimage.metrics import structural_similarity as ssim

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Layer, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing import image
from sklearn.metrics import confusion_matrix, accuracy_score

from roboflow import Roboflow

In [ ]:
# Dataset Setup

rf = Roboflow(api_key="your-api-key")
project = rf.workspace("roboflow-universe-projects").project("banana-ripeness-classification")
version = project.version(6)
dataset = version.download("folder")

DATASET_PATH = dataset.location
TRAIN_DIR = DATASET_PATH + "/train"
VAL_DIR   = DATASET_PATH + "/valid"
TEST_DIR  = DATASET_PATH + "/test"

In [ ]:
# Exploratory Data Analysis

classes = os.listdir(TRAIN_DIR)

class_counts = {}
for cls in classes:
    class_dir = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(class_dir):
        class_counts[cls] = len(os.listdir(class_dir))

plt.figure(figsize=(8, 5))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title("Banana Ripeness Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.show()

plt.figure(figsize=(12, 8))
for cls in classes:
    class_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(class_dir):
        continue
    images = os.listdir(class_dir)[:50]
    hist_sum = np.zeros((256, 3))
    for img_name in images:
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        for i in range(3):
            hist = cv2.calcHist([img], [i], None, [256], [0, 256])
            hist_sum[:, i] += hist[:, 0]
    hist_sum /= len(images)
    plt.plot(hist_sum[:, 0], label=f"{cls} - Blue")
    plt.plot(hist_sum[:, 1], label=f"{cls} - Green")
    plt.plot(hist_sum[:, 2], label=f"{cls} - Red")

plt.title("Average Color Histogram per Class")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Cosine Similarity Between Classes

features = defaultdict(list)
for cls in classes:
    class_dir = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(class_dir):
        continue
    images = os.listdir(class_dir)[:30]
    for img_name in images:
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (64, 64))
        features[cls].append(img.flatten())

class_vectors = {cls: np.mean(vecs, axis=0) for cls, vecs in features.items()}
class_list = list(class_vectors.keys())

similarity_matrix = np.zeros((len(class_list), len(class_list)))
for i, c1 in enumerate(class_list):
    for j, c2 in enumerate(class_list):
        sim = cosine_similarity([class_vectors[c1]], [class_vectors[c2]])
        similarity_matrix[i, j] = sim[0][0]

plt.figure(figsize=(6, 5))
sns.heatmap(similarity_matrix, xticklabels=class_list, yticklabels=class_list, annot=True, cmap="YlGnBu")
plt.title("Cosine Similarity Between Banana Ripeness Classes")
plt.show()

In [ ]:
# SSIM Structural Similarity

def compute_ssim(img1, img2):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    return ssim(gray1, gray2)

ssim_matrix = np.zeros((len(class_list), len(class_list)))
for i, c1 in enumerate(class_list):
    for j, c2 in enumerate(class_list):
        dir1 = os.path.join(TRAIN_DIR, c1)
        dir2 = os.path.join(TRAIN_DIR, c2)
        img1 = cv2.imread(os.path.join(dir1, os.listdir(dir1)[0]))
        img2 = cv2.imread(os.path.join(dir2, os.listdir(dir2)[0]))
        if img1 is None or img2 is None:
            continue
        img1 = cv2.resize(img1, (128, 128))
        img2 = cv2.resize(img2, (128, 128))
        ssim_matrix[i, j] = compute_ssim(img1, img2)

plt.figure(figsize=(6, 5))
sns.heatmap(ssim_matrix, xticklabels=class_list, yticklabels=class_list, annot=True, cmap="coolwarm")
plt.title("SSIM Structural Similarity Between Banana Ripeness Classes")
plt.show()

In [ ]:
# Data Generators

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 50

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.9, 1.1],
    channel_shift_range=10.0
)

val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse"
)

In [ ]:
# Custom Layers

class RGB2LAB(Layer):
    def call(self, inputs):
        rgb = inputs
        mask = tf.cast(rgb > 0.04045, tf.float32)
        rgb = tf.where(mask > 0, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)

        r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]

        x = r * 0.4124 + g * 0.3576 + b * 0.1805
        y = r * 0.2126 + g * 0.7152 + b * 0.0722
        z = r * 0.0193 + g * 0.1192 + b * 0.9505

        x /= 0.95047
        y /= 1.00000
        z /= 1.08883

        def f(t):
            delta = 6 / 29
            return tf.where(t > delta ** 3, tf.pow(t, 1/3), (t / (3 * delta ** 2)) + 4 / 29)

        fx, fy, fz = f(x), f(y), f(z)

        L = 116 * fy - 16
        A = 500 * (fx - fy)
        B = 200 * (fy - fz)

        return tf.stack([L, A, B], axis=-1)


class ColorRobustLayer(Layer):
    def call(self, inputs):
        L = inputs[..., 0] / 100.0
        A = inputs[..., 1] / 128.0
        B = inputs[..., 2] / 128.0
        lab = tf.stack([L, A, B], axis=-1)
        noise = tf.random.normal(tf.shape(lab), mean=0.0, stddev=0.02)
        return lab + noise

In [ ]:
# Model Architecture

inputs = Input(shape=(224, 224, 3))
x = RGB2LAB()(inputs)
x = ColorRobustLayer()(x)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=x)

for layer in base_model.layers[:-50]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(3, activation="sigmoid", name="ordinal_output")(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

In [ ]:
# Ordinal Encoding

def ordinal_encode(y):
    return np.array([1 if y > i else 0 for i in range(3)])

def ordinal_generator(generator):
    for x, y in generator:
        y_ord = np.array([ordinal_encode(label) for label in y])
        yield x, y_ord

train_gen_ord = ordinal_generator(train_data)
val_gen_ord = ordinal_generator(val_data)

In [ ]:
# Custom Loss

def ordinal_loss(y_true, y_pred):
    loss_bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    true_stage = tf.reduce_sum(y_true, axis=1)
    pred_stage = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=1)
    loss_dist = tf.reduce_mean(tf.abs(pred_stage - true_stage))
    return loss_bce + 0.1 * loss_dist

In [ ]:
# Compile and Train

steps_per_epoch = len(train_data)

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-4,
    decay_steps=steps_per_epoch * 20
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=ordinal_loss,
    metrics=["accuracy", tf.keras.metrics.MeanAbsoluteError(name="MAE")]
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/best_mobilenetv2_run3.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

model.fit(
    train_gen_ord,
    validation_data=val_gen_ord,
    steps_per_epoch=len(train_data),
    validation_steps=len(val_data),
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

In [ ]:
# Evaluation

model_run3 = tf.keras.models.load_model(
    "/content/drive/MyDrive/best_mobilenetv2_run3.keras",
    custom_objects={
        "RGB2LAB": RGB2LAB,
        "ColorRobustLayer": ColorRobustLayer,
        "ordinal_loss": ordinal_loss
    }
)

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=64,
    class_mode="sparse",
    shuffle=False
)

y_pred = model_run3.predict(test_generator)
y_pred_classes = np.sum(y_pred > 0.5, axis=1)
y_true = test_generator.classes

accuracy = accuracy_score(y_true, y_pred_classes)
mare = np.mean(np.abs(y_true - y_pred_classes))
adj_correct = sum(1 for t, p in zip(y_true, y_pred_classes) if abs(t - p) <= 1)
adjacent_accuracy = adj_correct / len(y_true)
cm = confusion_matrix(y_true, y_pred_classes)

print("Accuracy:", accuracy)
print("MARE:", mare)
print("Adjacent Accuracy:", adjacent_accuracy)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Banana Ripeness Confusion Matrix")
plt.show()

In [ ]:
# Lighting Robustness Study

def create_dataset(mode):
    X, y = [], []
    for class_name, class_index in test_generator.class_indices.items():
        class_path = os.path.join(TEST_DIR, class_name)
        for img_name in os.listdir(class_path):
            img = cv2.imread(os.path.join(class_path, img_name))
            img = cv2.resize(img, (224, 224))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
            if mode == "dark":
                img = img * 0.5
            elif mode == "bright":
                img = np.clip(img * 1.5, 0, 1)
            elif mode == "warm":
                img[:, :, 0] = np.clip(img[:, :, 0] * 1.2, 0, 1)
                img[:, :, 2] = np.clip(img[:, :, 2] * 0.8, 0, 1)
            X.append(img)
            y.append(class_index)
    return np.array(X), np.array(y)

base_acc = accuracy_score(y_true, y_pred_classes)

X_dark, y_dark = create_dataset("dark")
X_bright, y_bright = create_dataset("bright")
X_warm, y_warm = create_dataset("warm")

dark_pred = np.sum(model_run3.predict(X_dark) > 0.5, axis=1)
bright_pred = np.sum(model_run3.predict(X_bright) > 0.5, axis=1)
warm_pred = np.sum(model_run3.predict(X_warm) > 0.5, axis=1)

dark_acc = accuracy_score(y_dark, dark_pred)
bright_acc = accuracy_score(y_bright, bright_pred)
warm_acc = accuracy_score(y_warm, warm_pred)

print("Original Accuracy:", base_acc)
print("Dark Accuracy:", dark_acc)
print("Bright Accuracy:", bright_acc)
print("Warm Light Accuracy:", warm_acc)
print("Drop (Dark):", base_acc - dark_acc)
print("Drop (Bright):", base_acc - bright_acc)
print("Drop (Warm):", base_acc - warm_acc)

In [ ]:
# Ablation Study

def create_ablation_dataset(mode):
    X, y = [], []
    for class_name, class_index in test_generator.class_indices.items():
        class_path = os.path.join(TEST_DIR, class_name)
        for img_name in os.listdir(class_path):
            img = cv2.imread(os.path.join(class_path, img_name))
            img = cv2.resize(img, (224, 224))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
            if mode == "dark":
                img = img * 0.5
            elif mode == "bright":
                img = np.clip(img * 1.5, 0, 1)
            elif mode == "blur":
                img = cv2.GaussianBlur(img, (9, 9), 0)
            X.append(img)
            y.append(class_index)
    return np.array(X), np.array(y)

X_dark, y_dark = create_ablation_dataset("dark")
X_bright, y_bright = create_ablation_dataset("bright")
X_blur, y_blur = create_ablation_dataset("blur")

dark_pred = np.sum(model_run3.predict(X_dark) > 0.5, axis=1)
bright_pred = np.sum(model_run3.predict(X_bright) > 0.5, axis=1)
blur_pred = np.sum(model_run3.predict(X_blur) > 0.5, axis=1)

dark_acc = accuracy_score(y_dark, dark_pred)
bright_acc = accuracy_score(y_bright, bright_pred)
blur_acc = accuracy_score(y_blur, blur_pred)

print("Dark Accuracy:", dark_acc)
print("Bright Accuracy:", bright_acc)
print("Blur Accuracy:", blur_acc)
print("Drop (Dark):", base_acc - dark_acc)
print("Drop (Bright):", base_acc - bright_acc)
print("Drop (Blur):", base_acc - blur_acc)

plt.figure(figsize=(6, 4))
plt.bar(["Original", "Dark", "Bright", "Blur"], [base_acc, dark_acc, bright_acc, blur_acc])
plt.ylabel("Accuracy")
plt.title("Ablation Study")
plt.show()

In [ ]:
# Grad-CAM Visualization

def grad_cam(img_name, model, layer_name="Conv_1"):
    img = image.load_img(img_name, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model([img_array])
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap)

    img = cv2.imread(img_name)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)

    superimposed = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
# Score-CAM Visualization

def score_cam(img_path, model, layer_name="Conv_1"):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    conv_layer = model.get_layer(layer_name)
    heatmap_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    conv_outputs, _ = heatmap_model([img_array])
    conv_outputs = conv_outputs.numpy()[0]

    weights = []
    for i in range(conv_outputs.shape[-1]):
        activation_map = conv_outputs[:, :, i]
        activation_map = cv2.resize(activation_map, (224, 224))
        min_val, max_val = activation_map.min(), activation_map.max()
        if max_val - min_val != 0:
            activation_map = (activation_map - min_val) / (max_val - min_val)
        else:
            activation_map = np.zeros_like(activation_map)
        masked_input = img_array * activation_map.reshape(1, 224, 224, 1)
        pred = model.predict(masked_input, verbose=0)
        weights.append(pred[0].max())

    weights = np.array(weights)
    cam = np.zeros(conv_outputs.shape[:2], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * conv_outputs[:, :, i]

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224, 224))
    if cam.max() != 0:
        cam = cam / cam.max()

    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    result = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
# Run Grad-CAM and Score-CAM on an uploaded image (Google Colab)

from google.colab import files
uploaded = files.upload()
img_name = list(uploaded.keys())[0]

grad_cam(img_name, model_run3, layer_name="Conv_1")
score_cam(img_name, model_run3, layer_name="Conv_1")